# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [ ]:

from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()

# Sesuaikan dengan nama file hasil unduhan Kaggle.
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indonesian_news.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Periksa nama file di folder data/raw/."
    )

FILE_SIZE_MB = DATA_PATH.stat().st_size / 1_000_000

print(f"Dataset: {DATA_PATH.name}")
print(f"Format: {DATA_PATH.suffix.lower()}")
print(f"Ukuran file: {FILE_SIZE_MB:.2f} MB")


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [ ]:

suffix = DATA_PATH.suffix.lower()

if suffix == ".parquet":
    lazy_data = pl.scan_parquet(DATA_PATH)
elif suffix == ".csv":
    lazy_data = pl.scan_csv(DATA_PATH, infer_schema_length=10000)
elif suffix in [".jsonl", ".ndjson"]:
    lazy_data = pl.scan_ndjson(DATA_PATH)
else:
    raise ValueError(
        f"Format {suffix} belum didukung oleh cell ini. "
        "Periksa format dataset yang diunduh."
    )

schema = lazy_data.collect_schema()

print("Struktur kolom dan tipe data:")
print(schema)


In [ ]:

null_expressions = [
    pl.col(column).null_count().alias(f"{column}_null")
    for column in schema.names()
]

ringkasan = lazy_data.select(
    [pl.len().alias("jumlah_baris")] + null_expressions
).collect()

ringkasan


In [ ]:

contoh_data = lazy_data.head(10).collect()
contoh_data


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:

connection = duckdb.connect()

if suffix == ".parquet":
    query = "SUMMARIZE SELECT * FROM read_parquet(?)"
elif suffix == ".csv":
    query = "SUMMARIZE SELECT * FROM read_csv_auto(?)"
elif suffix in [".jsonl", ".ndjson"]:
    query = "SUMMARIZE SELECT * FROM read_json_auto(?)"

profil_duckdb = connection.execute(
    query, [str(DATA_PATH)]
).fetchdf()

profil_duckdb


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

# Catatan Milestone 1

### 1. Ukuran Data

Dataset yang digunakan adalah **Indonesian News Dataset** yang diperoleh dari Kaggle. Ukuran file dataset mentah diperoleh melalui pemeriksaan file sebelum dilakukan pengolahan data.

### 2. Jumlah Baris

Jumlah baris dan kolom dataset diperoleh dari hasil profiling menggunakan Polars. Informasi ini digunakan untuk mengetahui ukuran awal dataset sebelum proses pembersihan data dilakukan.

### 3. Tipe Kolom

Pemeriksaan skema menggunakan Polars dilakukan untuk mengetahui tipe data pada setiap kolom. Kolom teks digunakan untuk menyimpan informasi berita, sedangkan kolom numerik atau tanggal, apabila tersedia, diperiksa kesesuaian tipe datanya.

### 4. Nilai Kosong (*Null*)

Nilai kosong diperiksa menggunakan Polars untuk mengetahui jumlah null pada setiap kolom. Kolom dengan jumlah nilai kosong terbanyak menjadi perhatian utama dalam proses Data Quality pada Milestone 2.

### 5. Anomali Awal

Potensi anomali yang perlu diperiksa lebih lanjut meliputi:

- Nilai kosong pada kolom penting.
- Artikel berita dengan judul atau isi yang kosong.
- Kemungkinan adanya artikel berita duplikat.
- Ketidakkonsistenan format teks.
- Ketidaksesuaian tipe data pada kolom tertentu.

Daftar tersebut merupakan kandidat pemeriksaan awal dan perlu dikonfirmasi berdasarkan hasil profiling.

### 6. Kesimpulan

Milestone 1 dilakukan untuk memahami struktur, ukuran, tipe data, dan kelengkapan Indonesian News Dataset. Profiling menggunakan Polars dan DuckDB memberikan gambaran awal mengenai karakteristik dataset yang menjadi dasar proses pembersihan data pada Milestone 2.